# Named Entity Recognition (NER)
**LING 350 · Guided lab with NLTK and spaCy**

In this lab, you will find named entities in text, inspect the labels assigned by two pretrained systems, and investigate their mistakes.

**By the end, you should be able to:**
- Explain the difference between a part-of-speech tag and an entity label.
- Run an NLTK pipeline: tokenize → assign POS tags → identify named entities.
- Use spaCy to extract and visualize entities.
- Explain entity boundaries, context effects, and the role of capitalization.
- Evaluate exact entity matches and add simple rules (optional extensions).

**Exercise** before running the next demonstration. Edit the answer cells by double-clicking them. Code with `TODO` comments already runs; you will modify it as instructed. A self-check section appears at the end.

## 1. Setup and the NER task

Run these setup cells from top to bottom. They install the libraries, download the resources, and load the English model. You can use this setup as provided; the main goal of this lab is to understand and use NER.

- `nltk` provides tokenization, POS tagging, and named entity recognition.
- `spacy` provides another pretrained system for processing text.
- `displacy` is included with spaCy and highlights named entities in the text.

In [ ]:
# Install the two libraries used in this lab.
# %pip installs into the notebook's Python environment.
%pip install -q "nltk==3.9.2" "spacy>=3.8,<3.9"

# Download spaCy's small English model.
!python -m spacy download en_core_web_sm

  Using cached en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Import the two language-processing libraries.
import nltk
import spacy

# Use spaCy's built-in tool to highlight named entities in text.
from spacy import displacy


In [ ]:
# Download the resources required by NLTK.
nltk.download("punkt_tab")                       # Tokenization
nltk.download("averaged_perceptron_tagger_eng")  # English POS tagging
nltk.download("maxent_ne_chunker_tab")           # Named entity recognition
nltk.download("words")                          # Word list used by NER

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


True

### What does NER predict?

NER predicts **where an entity mention starts and ends** and **what type of entity it is**. A mention can span several tokens: `Tim Cook` is one person mention even though it contains two words.

In these systems, dates and amounts are also treated as entity types. An ordinary noun such as `headquarters` is not automatically an entity mention.

| Meaning | spaCy label | Example |
|---|---|---|
| Person | `PERSON` | Tim Cook |
| Organization | `ORG` | Apple Inc. |
| Country, city, or state | `GPE` | California |
| Other geographic location | `LOC` | Pacific Ocean |
| Date | `DATE` | 2019 |
| Time | `TIME` | 3 p.m. |
| Money | `MONEY` | $20 |
| Percentage | `PERCENT` | 25% |
| Named product | `PRODUCT` | iPhone 11 |

These are **annotation examples**, not promises about what the model will predict. NLTK's default chunker uses a different label inventory: for example, it uses `ORGANIZATION` where spaCy uses `ORG`. It does not offer all of the spaCy categories above.

**POS tagging versus NER:** a POS tag describes a word's grammatical category; an entity label describes the type of a mention in context. In `Tim Cook`, both words can receive a proper-noun POS tag, while the whole phrase receives one `PERSON` entity label.

In [ ]:
# Keep the original example, including capitalization and punctuation.
text_example = (
    "In 2019, Apple Inc. announced the launch of the iPhone 11 at their "
    "headquarters in Cupertino, California, with Tim Cook, the CEO, "
    "presenting the new features."
)
print(text_example)

In 2019, Apple Inc. announced the launch of the iPhone 11 at their headquarters in Cupertino, California, with Tim Cook, the CEO, presenting the new features.


### Exercise 1 — Be the annotator

List the entity mentions you expect in `text_example`. Use the labels in the table above.


**Your answer:**

| Entity text | Label |
|---|---|
| *Add your entity* | |
|Tim Cook |PERSON |
| | |
| | |

**Explanation:** *Write here.*

## 2. NER with NLTK

We will make the steps visible: **text → tokens → POS-tagged tokens → named entities**.

In [ ]:
# Compare whitespace splitting with NLTK's tokenizer.
split_tokens = text_example.split()
word_tokens = nltk.word_tokenize(text_example)

print("Whitespace split:")
print(split_tokens)
print("\nNLTK tokens:")
print(word_tokens)

Whitespace split:
['In', '2019,', 'Apple', 'Inc.', 'announced', 'the', 'launch', 'of', 'the', 'iPhone', '11', 'at', 'their', 'headquarters', 'in', 'Cupertino,', 'California,', 'with', 'Tim', 'Cook,', 'the', 'CEO,', 'presenting', 'the', 'new', 'features.']

NLTK tokens:
['In', '2019', ',', 'Apple', 'Inc.', 'announced', 'the', 'launch', 'of', 'the', 'iPhone', '11', 'at', 'their', 'headquarters', 'in', 'Cupertino', ',', 'California', ',', 'with', 'Tim', 'Cook', ',', 'the', 'CEO', ',', 'presenting', 'the', 'new', 'features', '.']


In [ ]:
# Assign a POS tag to each token using NLTK's pretrained English tagger.
# Each result is a tuple: (token, POS tag).
tagged = nltk.pos_tag(word_tokens)

for word, pos in tagged:
    print(f"{word} \t {pos}")

In 	 IN
2019 	 CD
, 	 ,
Apple 	 NNP
Inc. 	 NNP
announced 	 VBD
the 	 DT
launch 	 NN
of 	 IN
the 	 DT
iPhone 	 NN
11 	 CD
at 	 IN
their 	 PRP$
headquarters 	 NNS
in 	 IN
Cupertino 	 NNP
, 	 ,
California 	 NNP
, 	 ,
with 	 IN
Tim 	 NNP
Cook 	 NNP
, 	 ,
the 	 DT
CEO 	 NNP
, 	 ,
presenting 	 VBG
the 	 DT
new 	 JJ
features 	 NNS
. 	 .


Common tags in this output include `NNP` (singular proper noun), `NN` (singular noun), `NNS` (plural noun), `VBD` (past-tense verb), and `CD` (cardinal number). These describe grammar; they are not entity categories.

In [ ]:
# Identify named entities from the POS-tagged tokens.
# binary=False keeps categories such as PERSON, GPE, and ORGANIZATION.
named_entities = nltk.ne_chunk(tagged, binary=False)

In [ ]:
print(named_entities)

(S
  In/IN
  2019/CD
  ,/,
  (PERSON Apple/NNP Inc./NNP)
  announced/VBD
  the/DT
  launch/NN
  of/IN
  the/DT
  (ORGANIZATION iPhone/NN)
  11/CD
  at/IN
  their/PRP$
  headquarters/NNS
  in/IN
  (GPE Cupertino/NNP)
  ,/,
  (GPE California/NNP)
  ,/,
  with/IN
  (PERSON Tim/NNP Cook/NNP)
  ,/,
  the/DT
  (ORGANIZATION CEO/NNP)
  ,/,
  presenting/VBG
  the/DT
  new/JJ
  features/NNS
  ./.)


### Exercise 2 — Inspect the pipeline

1. Find two tokens that differ between `.split()` and `nltk.word_tokenize()` in the original example.
2. Compare the NLTK entity list with your annotations. Find one correct prediction and one missed entity, wrong label, or incorrect boundary. If a type is absent from NLTK's label inventory, distinguish that limitation from a mistake within a supported type.
3. Why is knowing that a word is `NNP` insufficient to decide the boundary of a named entity?
4. Replace `practice_text` below with a sentence containing a person's full name and a place with more than one word. Run the pipeline and inspect the boundaries.

**Your answers:** *Write here.*

In [ ]:
# TODO: Replace this sentence with your own example.
practice_text = "Barack Obama visited New York."

# Apply the same three NLTK steps to your new sentence.
practice_tokens = nltk.word_tokenize(practice_text)
practice_tags = nltk.pos_tag(practice_tokens)
practice_results = nltk.ne_chunk(practice_tags)

print(practice_results)

(S
  (PERSON Barack/NNP)
  (PERSON Obama/NNP)
  visited/VBD
  (GPE New/NNP York/NNP)
  ./.)


## 3. NER with spaCy

`nlp` is the loaded language-processing pipeline. Calling `nlp(text)` processes the text and returns a **Doc** object containing tokens and predicted annotations. `doc.ents` contains the predicted entity **spans** (contiguous sequences of tokens).

The underscore in attributes such as `label_` requests a readable string label. For example, `ent.label_` can be `"PERSON"`.

In [ ]:
# Load the English model that we downloaded above.
# We will use nlp(text) to process text throughout this notebook.
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Process the same original example so we can compare the two systems.
doc = nlp(text_example)

In [ ]:
# doc is a spaCy Doc object, not a string. It stores the text along with the linguistic information spaCy predicts.
print(type(doc))

<class 'spacy.tokens.doc.Doc'>


In [ ]:
print(doc.text)  # The original text, as a string

In 2019, Apple Inc. announced the launch of the iPhone 11 at their headquarters in Cupertino, California, with Tim Cook, the CEO, presenting the new features.


In [ ]:
print(doc.ents) # The predicted named entities

(2019, Apple Inc., Cupertino, California, Tim Cook)


In [ ]:
# Build a list of (entity text, entity label) pairs.
spacy_entities = []
for ent in doc.ents:
    spacy_entities.append((ent.text, ent.label_))

print(spacy_entities)

[('2019', 'DATE'), ('Apple Inc.', 'ORG'), ('Cupertino', 'GPE'), ('California', 'GPE'), ('Tim Cook', 'PERSON')]


In [ ]:
# Highlight entity spans in the original text inside the notebook.
# style="ent" selects named entities; jupyter=True displays inline.
displacy.render(doc, style="ent", jupyter=True)

In [ ]:
# This lists the categories the loaded spaCy NER component can predict.
print("\nspaCy NER labels:", nlp.get_pipe("ner").labels)


spaCy NER labels: ('CARDINAL', 'DATE', 'EVENT', 'FAC', 'GPE', 'LANGUAGE', 'LAW', 'LOC', 'MONEY', 'NORP', 'ORDINAL', 'ORG', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY', 'TIME', 'WORK_OF_ART')


### Exercise 3 — Extract selected entity types

The starter code below extracts people. Modify it to extract **both people and organizations**. Then replace `my_text` with two or three sentences of your own containing at least a person, an organization, and a place. Keep normal capitalization and punctuation.

1. Show all predicted entities and your filtered result.
2. Compare the result with what you intended the entities to mean. Identify any mistake or omission.
3. In the original Apple example, do NLTK and spaCy disagree about any mention's boundaries or label? Remember that `ORG` and `ORGANIZATION` are different names for a similar category.
4. Does a longer entity list necessarily mean a better system? Explain.

These short classroom examples are not enough to rank the systems' overall accuracy.

**Your answers:** *Write here.*

In [ ]:
# TODO: Replace this fictional practice text with your own sentences.
my_text = "Maya Chen works at Microsoft in Seattle. She met Daniel Kim on Monday."

# Process your text with the pretrained pipeline.
my_doc = nlp(my_text)
print("All predictions:", [(ent.text, ent.label_) for ent in my_doc.ents])

# TODO: Add the organization label to this set.
# A set stores the labels we want to keep; `in` checks membership.
target_labels = {"PERSON"}

selected_entities = []
for ent in my_doc.ents:
    if ent.label_ in target_labels:
        selected_entities.append((ent.text, ent.label_))

print("Selected entities:", selected_entities)

All predictions: [('Maya Chen', 'PERSON'), ('Microsoft', 'ORG'), ('Seattle', 'GPE'), ('Daniel Kim', 'PERSON'), ('Monday', 'DATE')]
Selected entities: [('Maya Chen', 'PERSON'), ('Daniel Kim', 'PERSON')]


## 4. Context and capitalization

NER often benefits from keeping capitalization and punctuation. Lowercasing, removing words, or stemming may remove useful clues and change entity boundaries.

### Exercise 4 — Predict, change one thing, compare

Before running the next cell, predict how `Apple`, `apple`, and `Jordan` should be interpreted in each sentence. Then compare your judgments with the output.

1. Does the model distinguish the company from the fruit? The person from the country?
2. In the capitalization comparison, which predictions change? If none change, report that result.
3. Add your own context pair using an ambiguous name or word, such as `Amazon`, `May`, or `Washington`.
4. Explain why the company/fruit sentences test contextual interpretation, while the original/lowercase pair gives a more controlled test of capitalization.

A model's prediction does not determine the correct interpretation. Use the sentence meaning to judge the output.

**Your predictions and observations:** *Write here.*

In [ ]:
# These are illustrative sentences for testing the model.
context_examples = [
    "Apple released a new phone.",
    "I ate an apple after lunch.",
    "Jordan scored a goal for his team.",
    "Amman is the capital of Jordan.",
]

# TODO: Add two sentences using another ambiguous name or word.
for sentence in context_examples:
    sentence_doc = nlp(sentence)
    print(sentence)
    print([(ent.text, ent.label_) for ent in sentence_doc.ents])
    print()

Apple released a new phone.
[('Apple', 'ORG')]

I ate an apple after lunch.
[]

Jordan scored a goal for his team.
[('Jordan', 'PERSON')]

Amman is the capital of Jordan.
[('Amman', 'GPE'), ('Jordan', 'GPE')]



In [ ]:
# Change capitalization while keeping the words and punctuation the same.
original_text = "Tim Cook works at Apple in California."
lowercase_text = original_text.lower()

for version_name, sentence in [("Original", original_text), ("Lowercase", lowercase_text)]:
    sentence_doc = nlp(sentence)
    print(version_name + ":", sentence)
    print([(ent.text, ent.label_) for ent in sentence_doc.ents])
    print()

Original: Tim Cook works at Apple in California.
[('Tim Cook', 'PERSON'), ('Apple', 'ORG'), ('California', 'GPE')]

Lowercase: tim cook works at apple in california.
[('tim cook', 'PERSON'), ('california', 'GPE')]



## 5. Entity boundaries and BIO tags

An entity label can apply to several consecutive tokens. **BIO** (also called IOB2) makes the boundaries explicit:

- **B**: beginning of an entity mention.
- **I**: inside the same entity mention, after its first token.
- **O**: outside an entity mention.

For a correctly identified `Tim Cook`, the tags are `B-PERSON` and `I-PERSON`. A one-token entity receives a `B` tag too. Adjacent mentions each start with their own `B` tag.

### Exercise 5 — Predict the token labels (5 minutes)

For `Tim Cook visited New York.`, assign a BIO tag to every token, including the period. Use `PERSON` and `GPE`. Then run the cell and compare your annotation with the model prediction.

| Token | Your BIO tag |
|---|---|
| Tim | |
| Cook | |
| visited | |
| New | |
| York | |
| . | |

**Follow-up:** If the model finds only `Cook` as `PERSON`, has it correctly identified the complete person mention? Explain.

**Your answer:** *Write here.*

In [ ]:
bio_doc = nlp("Tim Cook visited New York.")

# POS and entity information are stored separately on each token.
print(f"{'Token':12} {'POS':8} BIO tag")
for token in bio_doc:
    if token.ent_iob_ == "O":
        bio_tag = "O"
    else:
        # Combine the boundary marker with the entity type, e.g. B-PERSON.
        bio_tag = token.ent_iob_ + "-" + token.ent_type_
    print(f"{token.text:12} {token.pos_:8} {bio_tag}")

Token        POS      BIO tag
Tim          PROPN    B-PERSON
Cook         PROPN    I-PERSON
visited      VERB     O
New          PROPN    B-GPE
York         PROPN    I-GPE
.            PUNCT    O


## 6. Optional extension — Evaluate entity predictions

We will use **exact span-and-label matching**. A prediction is correct only when its start position, end position, and entity type all match the gold annotation (our reference answer).

- **True positive (TP):** a predicted entity exactly matches a gold entity.
- **False positive (FP):** a predicted entity has no exact match in the gold annotations.
- **False negative (FN):** a gold entity has no exact match in the predictions.

A wrong label or wrong boundary creates **one FP and one FN** in this evaluation. A correct prediction of `PERSON` on only `Einstein` does not exactly match `Albert Einstein`.

$$\mathrm{Precision}=\frac{TP}{TP+FP}\qquad
\mathrm{Recall}=\frac{TP}{TP+FN}\qquad
F_1=\frac{2TP}{2TP+FP+FN}$$

For the following exercise, use these annotations:

**Text:** `Marie Curie visited Paris and met Albert Einstein in Berlin.`

| Gold entity | Label |
|---|---|
| Marie Curie | PERSON |
| Paris | GPE |
| Albert Einstein | PERSON |
| Berlin | GPE |

**Deliberately imperfect predictions for practice:** `Marie Curie / PERSON`, `Paris / LOC`, `Einstein / PERSON`. These are supplied examples, not output from spaCy.

### Exercise 6 — Calculate precision, recall, and F1

1. Count TP, FP, and FN by hand.
2. Calculate precision, recall, and F1 before running the scoring cell.
3. Explain why labeling `Paris` as `LOC` counts as an error under the specified annotation scheme.
4. After scoring the actual spaCy predictions below, explain why performance on one sentence cannot establish overall model quality.

**Your answers:** TP = ___; FP = ___; FN = ___; precision = ___; recall = ___; F1 = ___.

**Explanation:** *Write here.*

In [ ]:
evaluation_text = "Marie Curie visited Paris and met Albert Einstein in Berlin."

# Represent each mention as (start character, end character, entity label).
# Positions distinguish separate mentions even when they have the same spelling.
def make_span(text, phrase, label, search_from=0):
    """Locate an exact phrase at or after search_from; return its labeled span."""
    start = text.index(phrase, search_from)
    end = start + len(phrase)
    return (start, end, label)

# Each phrase occurs once in this exercise.
# For repeated names, give search_from or specify each occurrence's offsets.
gold_spans = {
    make_span(evaluation_text, "Marie Curie", "PERSON"),
    make_span(evaluation_text, "Paris", "GPE"),
    make_span(evaluation_text, "Albert Einstein", "PERSON"),
    make_span(evaluation_text, "Berlin", "GPE"),
}

# These are the deliberately imperfect predictions from the exercise.
practice_predictions = {
    make_span(evaluation_text, "Marie Curie", "PERSON"),
    make_span(evaluation_text, "Paris", "LOC"),
    make_span(evaluation_text, "Einstein", "PERSON"),
}

In [ ]:
# This helper compares sets of complete labeled spans.
def score_entities(gold, predicted):
    gold = set(gold)
    predicted = set(predicted)

    # & means set intersection; - means set difference.
    tp = len(gold & predicted)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    # Return 0 if a denominator is zero, rather than dividing by zero.
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0

    return {"TP": tp, "FP": fp, "FN": fn,
            "precision": precision, "recall": recall, "F1": f1}

# Compare these results with your hand calculations.
practice_scores = score_entities(gold_spans, practice_predictions)
for metric, value in practice_scores.items():
    print(f"{metric}: {value:.3f}")

TP: 1.000
FP: 2.000
FN: 3.000
precision: 0.333
recall: 0.250
F1: 0.286


In [ ]:
# Now evaluate actual spaCy predictions against the same reference annotations.
evaluation_doc = nlp(evaluation_text)
model_spans = {
    (ent.start_char, ent.end_char, ent.label_)
    for ent in evaluation_doc.ents
}

print("Model predictions:", [(ent.text, ent.label_) for ent in evaluation_doc.ents])
print("Scores:", score_entities(gold_spans, model_spans))

# Translate unmatched offsets back into readable text for error analysis.
for title, spans in [
    ("Unmatched predictions (FP)", model_spans - gold_spans),
    ("Unmatched gold entities (FN)", gold_spans - model_spans),
]:
    print("\n" + title)
    if not spans:
        print("None")
    for start, end, label in sorted(spans):
        print(repr(evaluation_text[start:end]), label, (start, end))

Model predictions: [('Marie Curie', 'PERSON'), ('Paris', 'GPE'), ('Albert Einstein', 'PERSON'), ('Berlin', 'GPE')]
Scores: {'TP': 4, 'FP': 0, 'FN': 0, 'precision': 1.0, 'recall': 1.0, 'F1': 1.0}

Unmatched predictions (FP)
None

Unmatched gold entities (FN)
None


## 7. Optional extension — Add rules for local names

A pretrained model may miss a campus abbreviation or a course name. An **EntityRuler** lets us specify known phrases and labels. Here we add `CSULB` as an organization and `LING 350` as a custom `COURSE` entity type.

These rules provide explicit instructions; they do not train the statistical model. We load a separate pipeline for this experiment so that the comparisons above still use the original model.

In [ ]:
# A fresh pipeline also makes this cell safe to rerun without duplicate components.
nlp_rules = spacy.load("en_core_web_sm")

# Run the ruler after NER. Matching rules can replace overlapping model entities.
ruler = nlp_rules.add_pipe(
    "entity_ruler", after="ner", config={"overwrite_ents": True}
)

# String patterns match tokenized phrases with their original capitalization.
patterns = [
    {"label": "ORG", "pattern": "CSULB"},
    {"label": "COURSE", "pattern": "LING 350"},
]
ruler.add_patterns(patterns)

campus_text = "Students at CSULB are taking LING 350 this semester."
print("Original model:", [(ent.text, ent.label_) for ent in nlp(campus_text).ents])
print("With rules:", [(ent.text, ent.label_) for ent in nlp_rules(campus_text).ents])

Original model: [('350', 'CARDINAL'), ('this semester', 'DATE')]
With rules: [('CSULB', 'ORG'), ('LING 350', 'COURSE'), ('this semester', 'DATE')]


### Exercise 7 — Extend and challenge a rule (10 minutes)

1. Add a phrase rule for another course, such as `LING 424`, in `patterns` above. Rerun that cell and test it below.
2. Compare the exact phrase `LING 350` with `ling 350` and `LING350`. Which forms does your rule match? The statistical model may still make other predictions.
3. Consider a rule that always labels `Apple` as `ORG`. Test it on `Apple is my favorite fruit.` Why might a rule override a correct model prediction or force a wrong interpretation?
4. Would improving these particular examples prove that the rules improve overall accuracy? What additional data would you need?

**Your answers:** *Write here.*

In [ ]:
# TODO: Modify this text to test the new rule you added.
new_campus_text = "Students at CSULB can take LING 350 and LING 424."
print([(ent.text, ent.label_) for ent in nlp_rules(new_campus_text).ents])

# Compare spelling/capitalization variants of the same course name.
for course_text in ["I take LING 350.", "I take ling 350.", "I take LING350."]:
    print(course_text, [(ent.text, ent.label_) for ent in nlp_rules(course_text).ents])

[('CSULB', 'ORG'), ('LING 350', 'COURSE'), ('424', 'CARDINAL')]
I take LING 350. [('LING 350', 'COURSE')]
I take ling 350. [('350', 'CARDINAL')]
I take LING350. []


In [ ]:
# TODO: Uncomment the next line to try an intentionally overgeneral rule.
# ruler.add_patterns([{"label": "ORG", "pattern": "Apple"}])

fruit_text = "Apple is my favorite fruit."
print("Original model:", [(ent.text, ent.label_) for ent in nlp(fruit_text).ents])
print("With rules:", [(ent.text, ent.label_) for ent in nlp_rules(fruit_text).ents])

Original model: [('Apple', 'ORG')]
With rules: [('Apple', 'ORG')]


## Self-check notes — read after attempting the exercises

**Exercise 1:** Under the annotation scheme used here, reasonable annotations are `2019 / DATE`, `Apple Inc. / ORG`, `iPhone 11 / PRODUCT`, `Cupertino / GPE`, `California / GPE`, and `Tim Cook / PERSON`. `CEO` is a role and `headquarters` is a common noun in this context. Models may miss or mislabel some of these mentions.

**Exercise 2:** Whitespace splitting leaves commas attached in examples such as `2019,` and `Cupertino,`; NLTK separates those commas. Proper names can name different kinds of entities, so an `NNP` tag alone cannot resolve the category. Check actual model output for boundary errors and unsupported types.

**Exercise 3:** Use `target_labels = {"PERSON", "ORG"}`. More predictions can include more false positives; a larger list is not proof of better accuracy. Compare both labels and complete spans.

**Exercise 4:** Intended interpretations are company, fruit (no named entity), person, and country, respectively. The model may not recover all of them. Lowercasing removes a potential clue; it need not change every prediction. The Apple examples differ in both context and capitalization, whereas the original/lowercase pair changes only capitalization.

**Exercise 5:** Expected tags are `Tim/B-PERSON`, `Cook/I-PERSON`, `visited/O`, `New/B-GPE`, `York/I-GPE`, `./O`. Finding only `Cook` gives an incomplete boundary.

**Exercise 6:** TP = 1, FP = 2, FN = 3. Precision = 1/3 ≈ 0.333; recall = 1/4 = 0.250; F1 = 2/7 ≈ 0.286. The wrong `Paris` label and incomplete `Einstein` boundary each contribute an FP and an FN. Missing `Berlin` contributes another FN. A broader evaluation needs many independently annotated examples from the intended use case.

**Exercise 7:** Add `{"label": "COURSE", "pattern": "LING 424"}` to `patterns`. The supplied string rule for `LING 350` does not match the lowercase or unspaced variants. A rule for every occurrence of `Apple` can mislabel the fruit. Evaluate changes on held-out examples, including negative cases; do not report performance only on examples used to design the rules.

## Troubleshooting and references

| Symptom | What to do |
|---|---|
| `NameError` for `nltk`, `spacy`, or `nlp` | Run the setup/import/model cells first. |
| NLTK `LookupError` | Rerun the resource-download cell with internet access. |
| spaCy `[E050]` model missing | Rerun the model-download command in the first code cell, then load the model again. |
| Import error after installation | Restart the notebook session and rerun from the imports cell. |
| Different entities from a classmate | Check that the text is identical; predictions can also vary with model versions. |
| No entities found | An empty list is a valid prediction. Check the sentence and discuss what was missed. |

Documentation:
- [NLTK entity chunking](https://www.nltk.org/api/nltk.chunk.html)
- [NLTK named-entity chunker implementation and resource use](https://www.nltk.org/_modules/nltk/chunk/named_entity.html)
- [spaCy model installation](https://spacy.io/usage/models)
- [spaCy named entities and linguistic features](https://spacy.io/usage/linguistic-features#named-entities)
- [spaCy EntityRuler](https://spacy.io/usage/rule-based-matching#entityruler)
